# 0.94645: four feature views beat the 0.94644 blend

For two days nothing moved the public 0.94644 rank-average — not re-weighting, not
six more public sources, not a from-scratch LightGBM at 0.94633
([measured here](https://www.kaggle.com/code/megayak/s6e9-0-94644-plus-a-from-scratch-lightgbm)).
Every single-model change I tried on top of that LightGBM (full-data refit,
lift-vs-original features, centred-window target rates, max_bin 8192, additive
constraints, a wider target-encoder cross-fit, an MLP, a one-hot logistic, CatBoost)
was worth ±0.00003 on the same frozen folds.

What did move it: **the same ideas built as four separate pipelines with different
feature sets, rank-averaged.** Not one model with more features — several models
that see the data differently.

| member | what differs | OOF AUC | vs A |
|---|---|---|---|
| A | LightGBM, triple target encoding + smooth keys + digit/quantised/frequency features, 3 seeds | 0.946281 | 1 |
| B | XGBoost on exactly A's features | 0.946258 | 0.9995 |
| C | no digit features; centred-window target rates (income v±2…200); lift vs the original; smoothing 2/30/300 | 0.946223 | 0.9989 |
| D | no exact-value key; //10, //50, //500, //5000 ladder; windows; smoothing auto/20/200 | 0.946077 | 0.9979 |
| **A+B+C+D** | 0.3 / 0.3 / 0.2 / 0.2, rank space | **0.946339** | |

| submission | public LB |
|---|---|
| A alone (single seed) | 0.94633 |
| **A+B+C+D ensemble alone** | **0.94639** |
| 0.94644 daily blend + 20% ensemble | 0.94645 |
| **0.94644 daily blend + 30% ensemble** | **0.94645** |
| 0.94644 daily blend + 50% ensemble | 0.94644 |
| 0.94644 daily blend + 10% of my MLP | 0.94641 |

Two things in that table are the whole lesson. **B adds nothing**: same features,
different algorithm, correlation 0.9995. **C and D add even though they are weaker**:
a member 0.0002 below A still helps at 20% once its correlation drops to 0.998.
Diversity comes from the feature view, not the learner. The MLP row is the other
half — a different family that is 0.001 weaker is *negative*.

This notebook is the daily blend with the ensemble at 30%. The four members' OOF
and test predictions are in a public dataset so you can re-weight or stack them:
[megayak/s6e9-four-feature-views-one-ensemble-oof](https://www.kaggle.com/datasets/megayak/s6e9-four-feature-views-one-ensemble-oof).

In [ ]:
import glob
import numpy as np
import pandas as pd
from scipy.stats import rankdata, spearmanr

def find_one(*pats):
    for p in pats:
        h = sorted(glob.glob(f"/kaggle/input/**/{p}", recursive=True))
        if h:
            return h[0]
    raise FileNotFoundError(" | ".join(pats))

ID, TARGET = "id", "Will_Buy_EV"
rank01 = lambda s: rankdata(s) / len(s)

def load(*pats, col=None):
    p = find_one(*pats); print("  ", p)
    d = pd.read_csv(p).sort_values(ID)
    return d[col if col else TARGET].to_numpy()

test_id = pd.read_csv(find_one("test.csv"), usecols=[ID]).sort_values(ID)[ID].to_numpy()
print("sources:")
nina     = load("ps-s6e9-h-blend-1/submission.csv", "nina2025/**/submission.csv")
zoomzoom = load("submission_curvature9_lgbpair05.csv")
kospintr = load("evehicle-stacked-lgbm-catb-xgb-hgbc-baseline/submission.csv", "kospintr/**/submission.csv")
mikhail  = load("electric-vehicle-purchases-single-xgb/submission.csv",
                "electric-vehicle-purchases-xgb/submission.csv", "mikhailnaumov/**/submission.csv")
najiama  = load("pure-lgbm-model-cv-0-94606-lb-0-94637/submission_LIGHTGBM.csv", "**/submission_LIGHTGBM.csv")
realmlp  = load("ps-s6-e9-realmlp-pytorch/submission.csv", "yekenot/**/submission.csv")
fourview = load("test_four_views.csv", col="ensemble_ABCD")

# talhatursun's daily recipe, unchanged
base  = 0.5 * rank01(nina) + 0.5 * rank01(zoomzoom)
E     = 0.90 * base + 0.05 * rank01(kospintr) + 0.05 * rank01(mikhail)
F     = 0.50 * rank01(E) + 0.50 * rank01(najiama)
daily = rank01(0.85 * rank01(F) + 0.15 * rank01(realmlp))

W = 0.3
final = rank01((1 - W) * daily + W * rank01(fourview))
print(f"\nfour-view ensemble vs daily blend  spearman = {spearmanr(rank01(fourview), daily).statistic:.5f}")
print(f"final vs daily blend               spearman = {spearmanr(final, daily).statistic:.5f}")
pd.DataFrame({ID: test_id, TARGET: final}).to_csv("submission.csv", index=False)
print(f"wrote submission.csv  ({1-W:.0%} daily + {W:.0%} four-view ensemble)")

## How the four members were built

All four use the same frozen partition — `StratifiedKFold(10, shuffle=True, random_state=42)`
on `train.csv` row order — so their out-of-fold predictions can be compared and blended
row for row. Every target statistic is fit inside the outer fold (sklearn `TargetEncoder`
with an inner 5-fold cross-fit for the training rows, full-fold statistics for the
validation and test rows). Fold predictions are pooled in rank space.

* **A** is the model in [One LightGBM From Raw Data](https://www.kaggle.com/code/megayak/s6e9-one-lightgbm-from-raw-data-cv-0-9463),
  averaged over three seeds.
* **B** is XGBoost (`hist`, depth 5, colsample 0.3, max_bin 1024) on the identical matrix.
* **C** drops the digit features and adds centred-window target rates — the smoothed buy
  rate over income `v±2, ±5, ±10, ±25, ±50, ±200` and commute `±1, ±3, ±10`, plus the same
  windows restricted to rows of the same (city, car type) — and lift/novelty of each value
  against the original 10k dataset, with smoothings 2/30/300 instead of auto/10/100.
* **D** removes the exact-income key entirely and encodes the `//10, //50, //500, //5000`
  ladder instead, with the windows and smoothings auto/20/200.

The ensemble weights (0.3/0.3/0.2/0.2) came from a coarse grid on OOF; the equal
average scores 0.946337, so they barely matter.

## What I would try next

The pace is about +0.00003 OOF per new feature view, and the OOF gain carried to the
board one-for-one this time (+0.00006 both). Views E and F (a different quantisation
ladder, a model with target rates as `init_score` instead of columns) are the obvious
next members. Anyone who wants to add their own view: the dataset above has the folds.

---

Credit: recipe and sources as in
[@talhatursun's daily blend](https://www.kaggle.com/code/talhatursun/s6e9-daily-rank-average-ensemble);
members [@nina2025](https://www.kaggle.com/code/nina2025/ps-s6e9-h-blend-1),
[@jazivxt](https://www.kaggle.com/datasets/jazivxt/s6e9-zoom-zoom-baseline),
[@kospintr](https://www.kaggle.com/code/kospintr/evehicle-stacked-lgbm-catb-xgb-hgbc-baseline),
[@mikhailnaumov](https://www.kaggle.com/code/mikhailnaumov/electric-vehicle-purchases-single-xgb),
[@najiama](https://www.kaggle.com/code/najiama/pure-lgbm-model-cv-0-94606-lb-0-94637),
[@yekenot](https://www.kaggle.com/code/yekenot/ps-s6-e9-realmlp-pytorch).
The window-rate idea follows the forum ablation by @tilii7 / @starkhushi / the "everything we
measured" thread; the four members and the measurements are mine. If this saved you
submissions, an upvote helps.